<a href="https://colab.research.google.com/github/BhupenderNayak/pytorch_projects/blob/main/DiabetesModel_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
from  sklearn.model_selection  import train_test_split
from  sklearn.preprocessing  import StandardScaler
from  sklearn.preprocessing import LabelEncoder
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset


In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)
#df = pd.read_csv(url , columns)mbmb

X = df.drop('Outcome' ,axis=1).values
y = df['Outcome'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)
y_train_t = torch.FloatTensor(y_train).view(-1 , 1)
y_test_t = torch.FloatTensor(y_test).view( -1 , 1)

In [ ]:
train_dataset = TensorDataset(X_train_t , y_train_t)
train_loader  = DataLoader( train_dataset , batch_size=32 , shuffle=True)

In [ ]:
 class DiabtesNN(nn.Module):
   def __init__(self):
    super(DiabtesNN , self).__init__()
    self.layer1 = nn.Linear(8 ,16)
    self.relu = nn.ReLU()
    self.output = nn.Linear(16 ,1)
   def forward(self , x):
    x = self.layer1(x)
    x = self.relu(x)
    x = self.output(x)
    return x
model = DiabtesNN()


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
epochs = 50

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:

        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")

model.eval()

with torch.no_grad():
    raw_outputs = model(X_test_t)

    probabilities = torch.sigmoid(raw_outputs)

    y_pred = (probabilities > 0.5).float()

    accuracy = (y_pred == y_test_t).float().mean()
    print(f"\nFinal Test Accuracy: {accuracy.item() * 100:.2f}%")

Epoch: 10/50 | Loss: 0.4286
Epoch: 20/50 | Loss: 0.4033
Epoch: 30/50 | Loss: 0.3928
Epoch: 40/50 | Loss: 0.3794
Epoch: 50/50 | Loss: 0.3798

Final Test Accuracy: 75.32%
